# DreamCanvas on Google Colab

Runs the website on a free GPU and gives you a public link to share.

1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells below in order. The first start downloads about 4 GB of model weights (3–5 minutes).

In [ ]:
# 0. Check for a GPU. If this fails: Runtime → Change runtime type → T4 GPU → Save, then run again from the top.
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE — running on CPU (slow). Fast SD-Turbo model will be used.')

In [ ]:
# 1. Upload dreamcanvas.zip (the project folder) when prompted
from google.colab import files
import os, zipfile
if not os.path.exists('genai-studio/app.py'):
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('.')
%cd genai-studio

In [ ]:
# 2. Install libraries (torch is already on Colab)
!pip -q install flask diffusers transformers accelerate safetensors

In [ ]:
# 3. Start the server in the background
# GPU → Stable Diffusion 1.5 (best quality). CPU → SD-Turbo (2 steps, ~15-30 s per image).
import subprocess, os, time
subprocess.run(['pkill', '-f', 'python app.py'])  # stop a server left over from an earlier run
time.sleep(2)
MODEL = 'sd15' if HAS_GPU else 'turbo'
env = dict(os.environ, GENAI_MODEL=MODEL, PORT='7860')
server = subprocess.Popen(['python', 'app.py'], env=env, stdout=open('server.log','w'), stderr=subprocess.STDOUT)
print('Starting with model:', MODEL, '— run the status cell at the bottom until it says ready')

In [ ]:
# 4. Get a public link with Cloudflare (no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, threading
subprocess.run(['pkill', '-f', 'cloudflared tunnel'])  # close an old tunnel from an earlier run
tunnel = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:7860'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('Open this link:', m.group(0)); break
# keep reading cloudflared's log in the background, otherwise the tunnel can freeze
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()

In [ ]:
# Optional: open privately inside Colab instead of a public link
from google.colab import output
output.serve_kernel_port_as_window(7860)

In [ ]:
# 5. Model status — re-run until status is 'ready' (first load downloads ~2-4 GB)
import json, urllib.request
print(json.load(urllib.request.urlopen('http://localhost:7860/api/health')))
!tail -n 5 server.log